1. load the dataset

In [1]:
# ============================================================================
# COMPLETE TABTRANSFORMER PIPELINE - FROM DATA LOADING TO MODEL TRAINING
# Each section is a separate cell - copy into your Jupyter notebook
# ============================================================================


"""
================================================================================
CELL 1: Install Required Libraries
================================================================================
"""
!pip install boto3
!pip install pandas
!pip install tensorflow
!pip install scikit-learn

print("✓ All packages installed!")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✓ All packages installed!


In [2]:
"""
================================================================================
CELL 2: Load Dataset from RunPod S3
================================================================================
"""
import boto3
import pandas as pd
from botocore.config import Config
from io import BytesIO

print("Loading dataset from RunPod S3...")

# ---- RunPod S3 location ----
BUCKET = "e9tcw5eupu"
KEY = "data/eff_training.csv"
ENDPOINT = "https://s3api-eu-ro-1.runpod.io"
REGION = "eu-ro-1"

# ---- Your RunPod S3 credentials ----
ACCESS_KEY = "user_37sKcYrvnk9UXaIY3B3Zr90MH0g"
SECRET_KEY = "rps_YW72UMRXEMRVC8A407OCL08J8G34U1B3QTNO1ETX18pa1n"

cfg = Config(
    region_name=REGION,
    signature_version="s3v4",
    s3={"addressing_style": "path"},
)

s3 = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    endpoint_url=ENDPOINT,
    config=cfg,
)

# Load CSV from RunPod S3 into a variable named `data`
obj = s3.get_object(Bucket=BUCKET, Key=KEY)
data = pd.read_csv(BytesIO(obj["Body"].read()))

print(f"✓ Data loaded successfully!")
print(f"  Data type: {type(data)}")
print(f"  Data shape: {data.shape}")
print(f"\nFirst few rows:")
print(data.head())


Loading dataset from RunPod S3...
✓ Data loaded successfully!
  Data type: <class 'pandas.DataFrame'>
  Data shape: (6134, 5138)

First few rows:
                           photo_id        f1        f2        f3        f4  \
0  6ab1d061f51c6079633aeceed2faeb0b  0.000068  0.108145 -0.138813  0.633156   
1  e94e2e05fb8b099955bbc4fa5ce81e22  0.020843  0.026005 -0.093442  0.736929   
2  ba6951a4f37fc9302243370e927a02e2  0.014542 -0.071332 -0.154407  0.577781   
3  947d16539d4702427aa74f737329ffb9  0.041775  0.075746 -0.128497  0.485010   
4  9326695bf62926ec22690f576a633bba  0.004397  0.058590 -0.154224  0.528140   

         f5        f6        f7        f8        f9  ...         hip  \
0  0.346266 -0.046055  0.016021 -0.058632  0.097968  ...  105.333900   
1  0.240569  0.089982 -0.112391  0.000435 -0.076110  ...  101.478989   
2  0.196485 -0.125341 -0.056713 -0.027295  0.094879  ...   97.488243   
3  0.120409  0.011227  0.017852 -0.089796 -0.011273  ...  120.586845   
4  0.290956 -0.1084

2. data preprocessing

2.1. categorical encoding for 'gender' feature

In [3]:
import pandas as pd                 # import pandas for data handling

data['gender'] = data['gender'].astype('category')  # convert 'gender' values to categorical type
data['gender'] = data['gender'].cat.codes           # replace 'gender' with its numeric category codes

In [4]:
data['gender'].head()

0    0
1    1
2    1
3    0
4    1
Name: gender, dtype: int8

In [5]:
#data['height_cm'].head()

2.2. define weight frequencies for class imbalance issue for weight_kg

In [6]:
import pandas as pd                     # import pandas for data handling
import numpy as np                      # import numpy to help with safe division

# Assume 'data' is your DataFrame and already loaded
#print("Preview of data:\n", data.head())  # print first few rows to check data
print("\nTotal samples in dataset:", len(data))  # print total number of rows

# -----------------------------
# 1. Create boolean masks for the three weight_kg classes
# -----------------------------
class_1_mask = data['weight_kg'] < 60                      # True where weight_kg is less than 60
class_2_mask = data['weight_kg'] > 100                     # True where weight_kg is greater than 100
class_3_mask = (data['weight_kg'] >= 60) & (data['weight_kg'] <= 100)  # True where weight is between 60 and 100

# -----------------------------
# 2. Calculate class frequencies (counts)
# -----------------------------
freq_class_1 = class_1_mask.sum()          # number of samples with weight_kg < 60
freq_class_2 = class_2_mask.sum()          # number of samples with weight_kg > 100
freq_class_3 = class_3_mask.sum()          # number of samples with 60 <= weight_kg <= 100

print("\nClass frequencies:")              # header for clarity
print("Class 1 (weight_kg < 60):", freq_class_1)   # print frequency of class 1
print("Class 2 (weight_kg > 100):", freq_class_2)  # print frequency of class 2
print("Class 3 (60 <= weight_kg <= 100):", freq_class_3)  # print frequency of class 3

# -----------------------------
# 3. Number of classes according to the strategy
# -----------------------------
num_classes = 3                             # we defined three classes by the rules above
print("\nNumber of classes:", num_classes)  # print number of classes

# -----------------------------
# 4. Compute inverse-frequency weights for each class
#    Formula: w = total_samples / (num_classes * class_frequency)
# -----------------------------
total_samples = len(data)                   # total number of rows in the dataset

def safe_weight(class_freq):                # helper function to avoid division by zero
    if class_freq == 0:                     # check if a class has zero samples
        return np.nan                       # return NaN if no samples exist for that class
    return total_samples / (num_classes * class_freq)  # apply weighting formula

weight_class_1 = safe_weight(freq_class_1)  # compute weight for class 1
weight_class_2 = safe_weight(freq_class_2)  # compute weight for class 2
weight_class_3 = safe_weight(freq_class_3)  # compute weight for class 3

print("\nClass weights (inverse frequency):")          # header for class weights
print("Weight for Class 1 (weight_kg < 60):", weight_class_1)   # print weight of class 1
print("Weight for Class 2 (weight_kg > 100):", weight_class_2)  # print weight of class 2
print("Weight for Class 3 (60 <= weight_kg <= 100):", weight_class_3)  # print weight of class 3



Total samples in dataset: 6134

Class frequencies:
Class 1 (weight_kg < 60): 1049
Class 2 (weight_kg > 100): 514
Class 3 (60 <= weight_kg <= 100): 4571

Number of classes: 3

Class weights (inverse frequency):
Weight for Class 1 (weight_kg < 60): 1.9491579281855735
Weight for Class 2 (weight_kg > 100): 3.9779507133592737
Weight for Class 3 (60 <= weight_kg <= 100): 0.4473127689054182


2.3. define weight frequencies for class imbalance issue for gender feature

In [7]:
import numpy as np                                      # import numpy for numeric utilities (like NaN)

print("Preview of gender column:\n", data['gender'].head())  # show first few gender values to inspect

# -----------------------------------
# 1. Calculate class frequencies for gender
# -----------------------------------
gender_counts = data['gender'].value_counts()           # count how many samples belong to each gender class

print("\nClass frequencies for gender:")                # header for class frequency output
for gender_class, freq in gender_counts.items():        # loop over each gender class and its frequency
    print(f"Class {gender_class}: {freq}")              # print the class label and its frequency

# -----------------------------------
# 2. Number of gender classes
# -----------------------------------
num_gender_classes = len(gender_counts)                 # compute how many distinct gender classes we have
print("\nNumber of gender classes:", num_gender_classes)  # print number of gender classes

# -----------------------------------
# 3. Compute inverse-frequency weights for each gender class
#    Formula: w = total_samples / (num_classes * class_frequency)
# -----------------------------------
total_samples = len(data)                               # total number of samples in the dataset

def safe_weight(class_freq):                            # define helper function to compute class weight safely
    if class_freq == 0:                                 # check for zero frequency to avoid division by zero
        return np.nan                                   # return NaN if a class somehow has zero samples
    return total_samples / (num_gender_classes * class_freq)  # apply the inverse-frequency weight formula

gender_weights = {}                                     # create an empty dictionary to store weights per class
for gender_class, freq in gender_counts.items():        # loop through each gender class and its frequency
    gender_weights[gender_class] = safe_weight(freq)    # compute and store the weight for this gender class

print("\nClass weights (inverse frequency) for gender:")  # header for weight output
for gender_class, weight in gender_weights.items():     # loop over each class and its weight
    print(f"Weight for class {gender_class}: {weight}") # print the computed weight for this gender class


Preview of gender column:
 0    0
1    1
2    1
3    0
4    1
Name: gender, dtype: int8

Class frequencies for gender:
Class 1: 3650
Class 0: 2484

Number of gender classes: 2

Class weights (inverse frequency) for gender:
Weight for class 1: 0.8402739726027397
Weight for class 0: 1.2347020933977455


2.4. weight frequencies for weight classes and gender classes

In [8]:
import numpy as np   # import numpy for numeric operations

# -------------------------------------------------
# 1. Store the already-computed weights for weight classes
#    (use the variables you created when handling weight_kg)
# -------------------------------------------------
weight_class_weights = {                          # dictionary to hold weight-class weights
    'weight_<60':  weight_class_1,                # weight for class: weight_kg < 60
    'weight_>100': weight_class_2,                # weight for class: weight_kg > 100
    'weight_60_100': weight_class_3               # weight for class: 60 <= weight_kg <= 100
}

print("Weight-class weights:", weight_class_weights)  # print weight-class weights to check

# gender_weights dict is assumed from previous step, e.g. {0: w0, 1: w1}
print("Gender-class weights:", gender_weights)        # print gender-class weights to check

# -------------------------------------------------
# 2. Multiply each gender class with each weight class
#    wi = w_weight * w_gender
# -------------------------------------------------
combined_weights = {}                                # dictionary to store combined class weights

print("\nCombined weights for each (weight_class, gender_class):")  # header
for w_label, w_w in weight_class_weights.items():    # loop over weight classes
    for g_label, w_g in gender_weights.items():      # loop over gender classes
        wi = w_w * w_g                               # multiply weight and gender class weights
        combined_weights[(w_label, g_label)] = wi    # store in dictionary
        print(f"{w_label} & gender {g_label}: {wi}") # print each combination

Weight-class weights: {'weight_<60': np.float64(1.9491579281855735), 'weight_>100': np.float64(3.9779507133592737), 'weight_60_100': np.float64(0.4473127689054182)}
Gender-class weights: {1: 0.8402739726027397, 0: 1.2347020933977455}

Combined weights for each (weight_class, gender_class):
weight_<60 & gender 1: 1.6378266755466175
weight_<60 & gender 0: 2.4066293742935403
weight_>100 & gender 1: 3.3425684487322993
weight_>100 & gender 0: 4.9115840732177505
weight_60_100 & gender 1: 0.37586527732408703
weight_60_100 & gender 0: 0.5522980121710618


2.5. create a dictionary for weights and row index

In [9]:
# Check current columns in the DataFrame
print("Columns before adding index column:\n", data.columns)

# Add a new column named 'index' with values from 0 to number_of_rows-1
data['index'] = range(len(data))

# Move 'index' to the front (optional, just for nicer viewing)
cols = ['index'] + [c for c in data.columns if c != 'index']  # build new column order
data = data[cols]                                            # reorder columns

# Show first few rows to verify the new indexing column
#print("\nDataFrame after adding 'index' column:\n", data.head())


Columns before adding index column:
 Index(['photo_id', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9',
       ...
       'hip', 'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh',
       'waist', 'wrist', 'gender', 'height_cm', 'weight_kg'],
      dtype='str', length=5138)


/tmp/ipykernel_1058791/1177919204.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['index'] = range(len(data))


In [10]:
import numpy as np               # import numpy for numeric operations
import pickle                    # import pickle to save Python objects

# -------------------------------------------------
# 0. We assume these already exist:
#    - weight_class_1, weight_class_2, weight_class_3
#    - gender_weights   (dict: {gender_class: weight})
# -------------------------------------------------

# create a dictionary of weight-class weights (same as before)
weight_class_weights = {         # dictionary mapping weight class labels to their weights
    'weight_<60':  weight_class_1,      # weight for class: weight_kg < 60
    'weight_>100': weight_class_2,      # weight for class: weight_kg > 100
    'weight_60_100': weight_class_3     # weight for class: 60 <= weight_kg <= 100
}

print("Weight-class weights:", weight_class_weights)  # print weight-class weights
print("Gender-class weights:", gender_weights)        # print gender-class weights

# -------------------------------------------------
# 1. Helper function to get the weight class label for a given weight_kg
# -------------------------------------------------
def get_weight_class(w):         # define a function that receives a single weight value
    if w < 60:                   # check if weight is less than 60
        return 'weight_<60'      # return label for class 1
    elif w > 100:                # check if weight is greater than 100
        return 'weight_>100'     # return label for class 2
    else:                        # otherwise weight is between 60 and 100 (inclusive)
        return 'weight_60_100'   # return label for class 3

# -------------------------------------------------
# 2. Build dictionary: keys = index values, values = combined weights
# -------------------------------------------------
final_weights = {}               # create empty dictionary to store final weights

print("\nBuilding final_weights dictionary...")  # message to track progress

for _, row in data.iterrows():   # loop over each row of the DataFrame
    idx_val = row['index']       # get the value from the 'index' column for this row
    gender_val = row['gender']   # get the gender class value for this row
    weight_val = row['weight_kg']# get the weight_kg value for this row

    w_class = get_weight_class(weight_val)        # determine weight class label from weight_kg
    w_weight = weight_class_weights[w_class]      # look up the weight-class weight
    w_gender = gender_weights[gender_val]         # look up the gender-class weight

    combined_w = w_weight * w_gender             # multiply to get combined weight w_i
    final_weights[idx_val] = combined_w          # store combined weight in dictionary with key=index

print("Number of entries in final_weights:", len(final_weights))  # print number of entries
print("First 5 items in final_weights:", list(final_weights.items())[:5])  # show first few items

# -------------------------------------------------
# 3. Check index 0: gender, weight_kg, and combined weight
# -------------------------------------------------
print("\nChecking entry with index 0...")        # message to show what we're doing

row0 = data.loc[data['index'] == 0].iloc[0]      # select the row where 'index' column equals 0

gender0 = row0['gender']                         # get gender value for index 0
weight0 = row0['weight_kg']                      # get weight_kg value for index 0
w_class0 = get_weight_class(weight0)             # get weight class label for index 0

w_weight0 = weight_class_weights[w_class0]       # get weight-class weight for index 0
w_gender0 = gender_weights[gender0]              # get gender-class weight for index 0
combined0_calc = w_weight0 * w_gender0           # calculate combined weight for index 0

print("Row 0 -> gender:", gender0)               # print gender class for index 0
print("Row 0 -> weight_kg:", weight0)            # print weight_kg for index 0
print("Row 0 -> weight class:", w_class0)        # print weight class label for index 0
print("w_weight for row 0:", w_weight0)          # print weight-class weight for index 0
print("w_gender for row 0:", w_gender0)          # print gender-class weight for index 0
print("Combined weight (calculated):", combined0_calc)        # print calculated combined weight
print("Combined weight from final_weights[0]:", final_weights[0])  # print value from dictionary

# -------------------------------------------------
# 4. Save final_weights dictionary as a pickle file
# -------------------------------------------------
print("\nSaving final_weights dictionary as pickle file...")   # message to track saving step

with open('final_weights.pkl', 'wb') as f:       # open a file named 'final_weights.pkl' in binary write mode
    pickle.dump(final_weights, f)                # write dictionary to the file using pickle

print("Dictionary saved to 'final_weights.pkl'.")# confirmation message


Weight-class weights: {'weight_<60': np.float64(1.9491579281855735), 'weight_>100': np.float64(3.9779507133592737), 'weight_60_100': np.float64(0.4473127689054182)}
Gender-class weights: {1: 0.8402739726027397, 0: 1.2347020933977455}

Building final_weights dictionary...
Number of entries in final_weights: 6134
First 5 items in final_weights: [(0, np.float64(0.5522980121710618)), (1, np.float64(0.37586527732408703)), (2, np.float64(0.37586527732408703)), (3, np.float64(0.5522980121710618)), (4, np.float64(0.37586527732408703))]

Checking entry with index 0...
Row 0 -> gender: 0
Row 0 -> weight_kg: 72.0
Row 0 -> weight class: weight_60_100
w_weight for row 0: 0.4473127689054182
w_gender for row 0: 1.2347020933977455
Combined weight (calculated): 0.5522980121710618
Combined weight from final_weights[0]: 0.5522980121710618

Saving final_weights dictionary as pickle file...
Dictionary saved to 'final_weights.pkl'.


In [11]:
data.head()

,index,photo_id,f1,f2,f3,f4,f5,f6,f7,f8,...,hip,leg-length,shoulder-breadth,shoulder-to-crotch,thigh,waist,wrist,gender,height_cm,weight_kg
0,0,6ab1d061f51c6079633aeceed2faeb0b,0.000068,0.108145,-0.138813,0.633156,0.346266,-0.046055,0.016021,-0.058632,...,105.333900,76.817467,35.362858,65.993683,54.459591,88.813789,16.764332,0,170.50,72.0
1,1,e94e2e05fb8b099955bbc4fa5ce81e22,0.020843,0.026005,-0.093442,0.736929,0.240569,0.089982,-0.112391,0.000435,...,101.478989,85.154358,37.256760,65.861588,52.773052,89.176338,15.690955,1,178.30,71.8
2,2,ba6951a4f37fc9302243370e927a02e2,0.014542,-0.071332,-0.154407,0.577781,0.196485,-0.125341,-0.056713,-0.027295,...,97.488243,81.410393,37.503147,66.042679,57.059261,82.201988,16.686253,1,176.25,76.5
3,3,947d16539d4702427aa74f737329ffb9,0.041775,0.075746,-0.128497,0.485010,0.120409,0.011227,0.017852,-0.089796,...,120.586845,69.361534,34.084633,60.413330,65.000000,102.323845,17.693762,0,152.10,88.9
4,4,9326695bf62926ec22690f576a633bba,0.004397,0.058590,-0.154224,0.528140,0.290956,-0.108486,-0.021441,-0.099909,...,110.543564,77.160583,38.086231,68.400543,57.172279,107.378578,16.594791,1,171.50,88.4


2.6. apply Standard scaling for body measurements and robust scaling for cnn extracted features

In [12]:
from sklearn.preprocessing import RobustScaler
import pickle

# ---------- scaling ----------
exclude_cols = ['photo_id', 'subject_id', 'index', 'gender']

# Dependent (target) columns
target_cols = [
    'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip',
    'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh',
    'waist', 'wrist', 'weight_kg'
]

# Keep only existing columns
target_cols = [c for c in target_cols if c in data.columns]

# Independent feature columns
feature_cols = [
    c for c in data.columns
    if c not in exclude_cols and c not in target_cols
]

# ---------- Initialize scalers ----------
features_scaler = RobustScaler()
targets_scaler = RobustScaler()

# ---------- Fit & transform ----------
if feature_cols:
    data[feature_cols] = features_scaler.fit_transform(data[feature_cols])

if target_cols:
    data[target_cols] = targets_scaler.fit_transform(data[target_cols])

# ---------- Save scalers locally ----------
with open("scaler_independent.pkl", "wb") as f:
    pickle.dump(features_scaler, f)

with open("scaler_dependent.pkl", "wb") as f:
    pickle.dump(targets_scaler, f)

print("Scalers saved successfully in current working directory.")

Scalers saved successfully in current working directory.


In [13]:
data.head()

,index,photo_id,f1,f2,f3,f4,f5,f6,f7,f8,...,hip,leg-length,shoulder-breadth,shoulder-to-crotch,thigh,waist,wrist,gender,height_cm,weight_kg
0,0,6ab1d061f51c6079633aeceed2faeb0b,-0.698223,1.138307,0.370072,0.202900,0.908705,0.577309,0.816638,0.388236,...,0.351615,-0.229978,-0.199958,0.104045,0.121176,0.033250,0.041179,0,-0.070053,-0.093750
1,1,e94e2e05fb8b099955bbc4fa5ce81e22,-0.293652,0.231949,2.686336,0.764837,-0.227144,2.625825,-0.796078,1.650451,...,-0.009067,1.025716,0.239110,0.086421,-0.140945,0.053732,-0.502395,1,0.476357,-0.102679
2,2,ba6951a4f37fc9302243370e927a02e2,-0.416348,-0.842102,-0.426003,-0.096957,-0.700891,-0.616604,-0.096826,1.057898,...,-0.382457,0.461804,0.296231,0.110582,0.525214,-0.340277,0.001639,1,0.332750,0.107143
3,3,947d16539d4702427aa74f737329ffb9,0.113970,0.780800,0.896750,-0.599313,-1.518424,1.439894,0.839622,-0.277711,...,1.778743,-1.352983,-0.496292,-0.640501,1.759358,0.796487,0.511857,0,-1.359019,0.660714
4,4,9326695bf62926ec22690f576a633bba,-0.613922,0.591494,-0.416686,-0.365764,0.314324,-0.362797,0.346151,-0.493812,...,0.839052,-0.178298,0.431409,0.425175,0.542779,1.082049,-0.044679,1,0.000000,0.638393


3. model training

3.1. split the data for independent and dependent features

In [14]:
# List of columns to be used as dependent (target) features
target_cols = [
    'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip',
    'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh',
    'waist', 'wrist', 'weight_kg'
]

# Select these columns from the DataFrame as the multi-target Y
Y = data[target_cols]                  # Y will hold all dependent variables for multi-target regression

print("Selected target columns:", target_cols)  # print which columns are used as targets
print("Shape of Y (samples, targets):", Y.shape)  # print shape to confirm dimensions

Selected target columns: ['ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip', 'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh', 'waist', 'wrist', 'weight_kg']
Shape of Y (samples, targets): (6134, 14)


In [15]:
# Columns to drop for building independent features (X)
drop_cols = ['photo_id', 'subject_id','index'] + target_cols   # combine ID columns with target columns

print("Columns to drop for X:\n", drop_cols)           # show which columns will be removed

# Create X by dropping ID columns and all target columns
X = data.drop(columns=drop_cols)                       # drop the unwanted columns to get independent features

print("\nShape of X (samples, independent features):", X.shape)  # print shape of X
#print("\nColumns in X:\n", X.columns.tolist())         # list all feature names in X

Columns to drop for X:
 ['photo_id', 'subject_id', 'index', 'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip', 'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh', 'waist', 'wrist', 'weight_kg']

Shape of X (samples, independent features): (6134, 5122)


3.2. import necessary libraries for model training

In [16]:
pip install matplotlib


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
# -----------------------------
# Imports
# -----------------------------
import numpy as np                          # numerical operations
import pickle                               # to load the final_weights.pkl file
import matplotlib.pyplot as plt             # for plotting loss curves

from sklearn.model_selection import train_test_split  # to create train/validation sets

import tensorflow as tf                     # main deep learning library
from tensorflow.keras.models import Sequential          # model container
from tensorflow.keras.layers import Dense, Dropout, InputLayer, LeakyReLU, Activation
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD, Adam, RMSprop

# -----------------------------
# Reproducibility (optional)
# -----------------------------
np.random.seed(42)                          # fix numpy random seed
tf.random.set_seed(42)                      # fix tensorflow random seed

# -----------------------------
# 1. Assume you already have:
#    - data DataFrame
#    - X (independent features)
#    - Y (multi-output targets)
# If not, you can recreate X, Y here.
# -----------------------------

# Example (uncomment if you want everything in one place):
# target_cols = [
#     'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip',
#     'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh',
#     'waist', 'wrist', 'weight_kg'
# ]
# Y = data[target_cols]                                            # select target columns
# drop_cols = ['photo_id', 'subject_id'] + target_cols             # columns not used as features
# X = data.drop(columns=drop_cols + ['index'])                     # drop also 'index' from features

print("Shape of X (features):", X.shape)           # show shape of feature matrix
print("Shape of Y (targets):", Y.shape)           # show shape of target matrix

# -----------------------------
# 2. Load final_weights.pkl (sample weights per index)
# -----------------------------
print("\nLoading final_weights.pkl ...")          # status message

with open('final_weights.pkl', 'rb') as f:        # open pickle file in read-binary mode
    final_weights_dict = pickle.load(f)           # load dictionary {index: weight}

print("Number of entries in final_weights_dict:", len(final_weights_dict))  # size of dictionary
print("First 5 entries in final_weights_dict:", list(final_weights_dict.items())[:5])  # preview


2026-02-20 10:50:50.552419: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Shape of X (features): (6134, 5122)
Shape of Y (targets): (6134, 14)

Loading final_weights.pkl ...
Number of entries in final_weights_dict: 6134
First 5 entries in final_weights_dict: [(0, np.float64(0.5522980121710618)), (1, np.float64(0.37586527732408703)), (2, np.float64(0.37586527732408703)), (3, np.float64(0.5522980121710618)), (4, np.float64(0.37586527732408703))]


3.3. convert weight dictionary for array to balance class imbalance of a regression problem

In [18]:
# -----------------------------
# 3. Build sample_weight array based on DataFrame 'index' column
# -----------------------------
print("\nBuilding sample_weight array ...")       # status message

# map each row's 'index' value to its weight in the dictionary
sample_weights = data['index'].map(final_weights_dict).values.astype('float32')

print("Sample weights shape:", sample_weights.shape)   # show shape of weight array
print("First 10 sample weights:", sample_weights[:10]) # preview some weights


Building sample_weight array ...
Sample weights shape: (6134,)
First 10 sample weights: [0.552298   0.37586528 0.37586528 0.552298   0.37586528 0.552298
 0.37586528 0.37586528 0.552298   0.37586528]


3.5. split the data into train and validation

In [19]:
# -----------------------------
# 4. Train/validation split (X, Y, and weights)
# -----------------------------
print("\nSplitting into train and validation sets ...")  # status message

X_train, X_val, Y_train, Y_val, w_train, w_val = train_test_split(
    X, Y, sample_weights,           # split features, targets, and weights together
    test_size=0.2,                  # 20% validation
    random_state=42,                # reproducible split
    shuffle=True                    # shuffle data before splitting
)

print("X_train shape:", X_train.shape)          # show training feature shape
print("Y_train shape:", Y_train.shape)          # show training target shape
print("X_val shape:", X_val.shape)              # show validation feature shape
print("Y_val shape:", Y_val.shape)              # show validation target shape


Splitting into train and validation sets ...
X_train shape: (4907, 5122)
Y_train shape: (4907, 14)
X_val shape: (1227, 5122)
Y_val shape: (1227, 14)


In [20]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.layers import LeakyReLU, Activation
from tensorflow.keras.optimizers import SGD, Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def build_model(num_hidden_layers, num_neurons, activation_name,
                learning_rate, optimizer_name, input_dim, output_dim,
                dropout_rate=0.3):

    model = Sequential()
    model.add(Input(shape=(input_dim,)))   # ✅ fixed deprecation warning

    for _ in range(num_hidden_layers):
        model.add(Dense(num_neurons))

        if activation_name.lower() == 'leakyrelu':
            model.add(LeakyReLU(negative_slope=0.1))
        elif activation_name.lower() == 'gelu':
            model.add(Activation(tf.keras.activations.gelu))
        elif activation_name.lower() == 'tanh':
            model.add(Activation('tanh'))
        else:
            raise ValueError(f"Unknown activation: {activation_name}")

        model.add(Dropout(dropout_rate))

    model.add(Dense(output_dim, activation='linear'))

    if optimizer_name.lower() == 'sgd':
        optimizer = SGD(learning_rate=learning_rate, momentum=0.9)
    elif optimizer_name.lower() == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name.lower() == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    # ✅ Added MAE metric
    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.MeanSquaredError(name="mse")
        ]
    )

    return model


def run_grid_search(X_train, Y_train, w_train, X_val, Y_val, w_val):

    input_dim = X_train.shape[1]
    output_dim = Y_train.shape[1]

    # 🔥 Parameter Grid
    num_hidden_layers_list = [9, 5, 7]
    num_neurons_list = [256, 128, 512]
    activation_list = ['tanh']
    learning_rates = [1e-4, 1e-3, 1e-2]
    optimizer_list = ['rmsprop']

    batch_size = 200
    num_epochs = 300

    best_val_loss = np.inf
    best_val_mae = np.inf
    best_params = None
    best_model = None

    total_combinations = (
        len(num_hidden_layers_list) *
        len(num_neurons_list) *
        len(activation_list) *
        len(learning_rates) *
        len(optimizer_list)
    )

    print(f"\nTotal combinations: {total_combinations}\n")

    combo = 0

    for num_hidden_layers in num_hidden_layers_list:
        for num_neurons in num_neurons_list:
            for activation_name in activation_list:
                for lr in learning_rates:
                    for optimizer_name in optimizer_list:

                        combo += 1
                        print(f"\nTraining {combo}/{total_combinations}")
                        print(f"Layers: {num_hidden_layers}, "
                              f"Neurons: {num_neurons}, "
                              f"Activation: {activation_name}, "
                              f"LR: {lr}, "
                              f"Optimizer: {optimizer_name}")

                        tf.keras.backend.clear_session()

                        model = build_model(
                            num_hidden_layers,
                            num_neurons,
                            activation_name,
                            lr,
                            optimizer_name,
                            input_dim,
                            output_dim
                        )

                        early_stop = EarlyStopping(
                            monitor='val_loss',
                            patience=15,
                            restore_best_weights=True,
                            verbose=0
                        )

                        lr_scheduler = ReduceLROnPlateau(
                            monitor='loss',
                            factor=0.5,
                            patience=5,
                            min_lr=1e-6,
                            verbose=0
                        )

                        history = model.fit(
                            X_train,
                            Y_train,
                            sample_weight=w_train,
                            validation_data=(X_val, Y_val, w_val),
                            epochs=num_epochs,
                            batch_size=batch_size,
                            callbacks=[early_stop, lr_scheduler],
                            verbose=1
                        )

                        min_val_loss = min(history.history['val_loss'])
                        min_val_mae = min(history.history['val_mae'])

                        print(f"Best val_loss: {min_val_loss:.6f}")
                        print(f"Best val_mae : {min_val_mae:.6f}")

                        if min_val_loss < best_val_loss:
                            print(">>> New best model found!")

                            best_val_loss = min_val_loss
                            best_val_mae = min_val_mae
                            best_params = {
                                "num_hidden_layers": num_hidden_layers,
                                "num_neurons": num_neurons,
                                "activation": activation_name,
                                "learning_rate": lr,
                                "optimizer": optimizer_name
                            }
                            best_model = model

    print("\n================ BEST MODEL =================")
    print(f"Best Validation Loss: {best_val_loss:.6f}")
    print(f"Best Validation MAE : {best_val_mae:.6f}")
    print("Best Hyperparameters:")
    for k, v in best_params.items():
        print(f"{k}: {v}")
    print("=============================================")

    best_model.save("eff_ANN_v22.h5")
    print("\nBest model saved as eff_ANN_v22.h5")


# ✅ Run after your train_test_split
run_grid_search(X_train, Y_train, w_train, X_val, Y_val, w_val)


Total combinations: 27


Training 1/27
Layers: 9, Neurons: 256, Activation: tanh, LR: 0.0001, Optimizer: rmsprop


2026-02-20 10:50:53.465089: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Epoch 1/300
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - loss: 1.0056 - mae: 0.7156 - mse: 0.8180 - val_loss: 0.5415 - val_mae: 0.6929 - val_mse: 0.7053 - learning_rate: 1.0000e-04
Epoch 2/300
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.7138 - mae: 0.6736 - mse: 0.7247 - val_loss: 0.5707 - val_mae: 0.7149 - val_mse: 0.7397 - learning_rate: 1.0000e-04
Epoch 3/300
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.6553 - mae: 0.6422 - mse: 0.6625 - val_loss: 0.5798 - val_mae: 0.7173 - val_mse: 0.7384 - learning_rate: 1.0000e-04
Epoch 4/300
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.6153 - mae: 0.6234 - mse: 0.6244 - val_loss: 0.5430 - val_mae: 0.6886 - val_mse: 0.6855 - learning_rate: 1.0000e-04
Epoch 5/300
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.5646 - mae: 0.6012 - mse: 0.5812 - val_loss: 0.5525 - val_mae: 0.6945 - val_mse: 0.6932 - learning_rate: 1.0000e-04
Epoch 6/300
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.5439 - mae: 0.5894 - mse: 0.5618 - val_loss: 0.5